# BTW Module 05: Advanced Downstream Analysis
**Downstream Bulk Transcriptomics Workbench (`btw`)**

สมุดงานตัวอย่างสาธิตการทำงานของ 3 โมดูลขั้นสูงใน **Part 5**:
1. **Gene Annotation & GTF Reference Mapping (FR-6):** การอ่านไฟล์ GTF (`gtfparse`), การดึงคู่แมปยีน ID ↔ Symbol, และการเพิ่ม Annotation เข้าตาราง Differential Expression (`annotate_de_results`)
2. **Batch Effect Correction (FR-7):** การขจัดตัวแปรแฝงทางเทคนิค (Batch Effects) ด้วย ComBat-Seq (`inmoose.pycombat`), การประเมินผลเชิงปริมาณ (Batch Silhouette Score), และการเปรียบเทียบการจัดกลุ่มตัวอย่างก่อน/หลังด้วย PCA แบบเคียงข้างกัน (`compare_pca_batch`)
3. **Co-expression Network & WGCNA (FR-8):** การคำนวณ Soft-thresholded Adjacency, Topological Overlap Matrix (TOM), การแบ่งกลุ่มยีน Co-expression Modules, การสกัด Module Eigengenes, การแปลงผลเข้าสู่ NetworkX (`module_to_networkx`), และการส่งออกเครือข่ายสำหรับโปรแกรม Cytoscape (`.sif` และ Edge Lists)

In [ ]:
import io
import os
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

import btw
from btw import set_seed, logger
from btw.de_analysis import run_de
from btw.annotation import (
    parse_gtf_file,
    create_gene_map_from_gtf,
    map_gene_ids,
    annotate_de_results,
)
from btw.batch_correction import (
    run_combat,
    compare_pca_batch,
    evaluate_batch_effect,
)
from btw.network import (
    WGCNAClusterResult,
    compute_adjacency,
    compute_tom,
    detect_coexpression_modules,
    module_to_networkx,
    export_cytoscape_sif,
    export_edge_list,
)
from btw.viz import set_publication_style, save_figure

# ตั้งค่าความเที่ยงตรงและธีมสไตล์ภาพระดับวารสาร
set_seed(42)
set_publication_style()
print(f"BTW version: {btw.__version__}")

## 1. Gene Annotation & GTF Reference Parsing (FR-6)
สาธิตการโหลด Annotation จากไฟล์ GTF หรือ Buffer และเชื่อมโยง Ensembl Gene IDs เข้ากับ Gene Symbols มาตรฐานลงในตาราง DE Results

In [ ]:
# 1. สร้างตัวอย่างข้อมูล GTF Reference
sample_gtf = """chr1\tHAVANA\tgene\t1000\t2500\t.\t+\t.\tgene_id "ENSG000001"; gene_name "TP53"; gene_biotype "protein_coding";
chr1\tHAVANA\tgene\t3000\t4500\t.\t-\t.\tgene_id "ENSG000002"; gene_name "BRCA1"; gene_biotype "protein_coding";
chr2\tHAVANA\tgene\t6000\t7200\t.\t+\t.\tgene_id "ENSG000003"; gene_name "MYC"; gene_biotype "protein_coding";
chr3\tHAVANA\tgene\t8000\t9100\t.\t+\t.\tgene_id "ENSG000004"; gene_name "EGFR"; gene_biotype "protein_coding";
"""

# 2. แปลง GTF เป็นตารางและสร้าง Gene Symbol Map
gtf_buf = io.StringIO(sample_gtf)
gtf_df = parse_gtf_file(gtf_buf, features=["gene"])
print("Parsed GTF table:")
print(gtf_df[["seqname", "feature", "gene_id", "gene_name", "gene_biotype"]])

gtf_buf.seek(0)
gene_map = create_gene_map_from_gtf(gtf_buf, from_attr="gene_id", to_attr="gene_name")
print(f"\nExtracted gene mapping ({len(gene_map)} genes):\n{gene_map}")

In [ ]:
# 3. นำเข้าผลลัพธ์ DE และเพิ่มคอลัมน์ Annotation Symbol
mock_de = pd.DataFrame(
    {
        "baseMean": [1200.5, 450.2, 890.1, 2300.0],
        "log2FoldChange": [2.35, -1.80, 1.45, -0.25],
        "pvalue": [1e-6, 2e-4, 5e-5, 0.45],
        "padj": [1e-5, 1e-3, 4e-4, 0.52],
    },
    index=["ENSG000001", "ENSG000002", "ENSG000003", "ENSG000004"],
)
mock_de.index.name = "gene_id"

annotated_de = annotate_de_results(mock_de, mapping_dict=gene_map.to_dict())
print("Annotated Differential Expression Table:")
annotated_de

## 2. Technical Batch Effect Correction via ComBat (FR-7)
สาธิตการกำจัดความแปรปรวนทางเทคนิค (เช่น วันที่เตรียมไลบรารี, เครื่องซีเควนซ์) ออกจากชุดข้อมูลนับยีน RNA-Seq ด้วย ComBat-Seq พร้อมตรวจสอบประสิทธิภาพผ่าน PCA และคำนวณ Batch Silhouette Score

In [ ]:
# 1. สร้างชุดข้อมูลตัวอย่างที่ได้รับผลกระทบจาก Technical Batch อย่างรุนแรง
np.random.seed(42)
n_genes, n_samples = 60, 6
genes = [f"GENE_{i:02d}" for i in range(n_genes)]
samples = [f"Sample_{i:02d}" for i in range(n_samples)]

# Count matrix พื้นฐาน
raw_counts = np.random.negative_binomial(n=25, p=0.08, size=(n_genes, n_samples))
# ปรับเพิ่มค่า count ให้ batch 2 (3 ตัวอย่างหลัง) เพื่อจำลอง Technical Batch Shift
raw_counts[:, 3:] += 160

counts_df = pd.DataFrame(raw_counts, index=genes, columns=samples)
meta_df = pd.DataFrame(
    {
        "batch": ["Batch_1", "Batch_1", "Batch_1", "Batch_2", "Batch_2", "Batch_2"],
        "condition": ["Control", "Treated", "Control", "Control", "Treated", "Treated"],
    },
    index=samples,
)

# 2. รัน ComBat-Seq สำหรับ RNA-Seq Counts พร้อมปกป้อง biological factor (condition)
corrected_counts = run_combat(
    data=counts_df,
    batch="batch",
    metadata=meta_df,
    biological_factor="condition",
    is_count=True,
)

# 3. เปรียบเทียบ PCA ก่อนและหลังการแก้ Batch Effect แบบเคียงข้างกัน
fig_dir = Path("reports/figures")
fig_dir.mkdir(parents=True, exist_ok=True)

fig_pca, (ax1, ax2), qc_metrics = compare_pca_batch(
    data_before=np.log2(counts_df + 1.0),
    data_after=np.log2(corrected_counts + 1.0),
    metadata=meta_df,
    batch_col="batch",
    condition_col="condition",
    title_prefix="Technical Batch Correction Diagnostic (ComBat-Seq)",
)
save_figure(fig_pca, fig_dir / "05_combat_pca_comparison.png", dpi=300)
plt.close(fig_pca)

print("Batch Effect Evaluation Metrics:")
print(f"- Batch Silhouette (Before): {qc_metrics['before']['batch_silhouette']:.4f}")
print(f"- Batch Silhouette (After):  {qc_metrics['after']['batch_silhouette']:.4f}")
print(f"- Reduction in Batch Separation: {qc_metrics['batch_silhouette_reduction']:.4f}")

## 3. Co-expression Networks & WGCNA (FR-8)
สาธิตการระบุ Co-expression Modules ผ่าน Topological Overlap Matrix (TOM), การสกัด Module Eigengenes, การเชื่อมโยงกับฟีโนไทป์, การแปลงเป็น NetworkX Graph, และการส่งออก Cytoscape SIF

In [ ]:
# 1. สร้างชุดข้อมูลตัวอย่างที่มี Co-expressed Gene Modules ชัดเจน
np.random.seed(42)
n_samples, n_genes = 12, 40
sample_ids = [f"Sample_{i:02d}" for i in range(n_samples)]
gene_ids = [f"Gene_{i:03d}" for i in range(n_genes)]

t = np.linspace(0, 2 * np.pi, n_samples)
sig1 = np.sin(t)
sig2 = np.cos(t)

expr = np.random.normal(0, 0.15, size=(n_samples, n_genes))
for j in range(15):
    expr[:, j] += sig1
for j in range(15, 30):
    expr[:, j] += sig2

expr_df = pd.DataFrame(expr, index=sample_ids, columns=gene_ids)
trait_df = pd.DataFrame({"Biomarker_Score": sig1 * 3.5}, index=sample_ids)

# 2. รัน WGCNA Co-expression Pipeline
wgcna_res = detect_coexpression_modules(
    data=expr_df,
    power=6,
    min_module_size=8,
    sample_metadata=trait_df,
    traits=["Biomarker_Score"],
)
print(wgcna_res.summary())

# แสดงความสัมพันธ์ระหว่าง Module Eigengenes กับ Trait
print("\nModule Eigengene Correlation with Biomarker Score:")
print(wgcna_res.module_trait_cor)

## 4. Graph Construction & Cytoscape SIF Export
แปลงโมดูลที่ค้นพบเป็น NetworkX Graph เพื่อคำนวณ Node Degree Centrality และส่งออกเป็น Cytoscape SIF / Edge list

In [ ]:
# 1. แปลงโมดูลแรกเป็น NetworkX Graph
top_module = wgcna_res.modules[0]
G = module_to_networkx(wgcna_res, module=top_module, threshold=0.08)
print(f"NetworkX Graph for Module '{top_module}':")
print(f"- Number of nodes: {G.number_of_nodes()}")
print(f"- Number of edges: {G.number_of_edges()}")

# 2. ส่งออกไฟล์เครือข่ายสำหรับ Cytoscape
net_dir = Path("reports/networks")
net_dir.mkdir(parents=True, exist_ok=True)

sif_path = export_cytoscape_sif(G, net_dir / f"wgcna_{top_module}.sif", interaction_type="coexpressed")
edge_path = export_edge_list(G, net_dir / f"wgcna_{top_module}_edges.tsv")

print(f"Successfully exported Cytoscape SIF: {sif_path}")
print(f"Successfully exported Edge List: {edge_path}")
print("Module 05 execution completed successfully!")